# Step 2: Split & Encode — Kickstarter campaigns, 2016–2025

**Goal:** split campaigns by launch year into four slices, prepare launch-day inputs, and one-hot encode them so training, validation and test have identical columns.

**Input:** `campaigns_clean.parquet` from Step 1 (273,163 campaigns, 22 columns).

**Output:** `split.zip` — `X_train` / `X_val` / `X_test` (191 model inputs each, including one-hot encodings) and `slice_text` / `slice_train` / `slice_val` / `slice_test` (full readable rows with `id`, `success`, `near_miss`, blurbs, aka things that are not model inputs).

| Slice | Years | Job | Campaigns | Near-misses |
|---|---|---|---|---|
| Text | 2016–18 | For Deep Learning Blurb model only | 100,240 | 1,673 |
| Train | 2019–23 | Baseline, logistic regression, gradient boosted models study here | 119,297 | 1,923 |
| Validation | 2024 | Settings, best model, cutoff | 26,739 | 541 |
| Test | 2025 | Final results, run once | 26,887 | 410 |

**To reproduce:** upload `campaigns_clean.parquet`, then run only the cells marked PIPELINE, top to bottom (a few minutes).

**Team rules:** every "most common" list and every median comes from training (2019–23) only. Columns starting with `post_` are never model inputs.

**Reference:** HW4 Section 2 (`build_features()`), HW2 Task 3 (overlap check).

### PIPELINE — Load the clean file
Loads Step 1's output and checks it arrived intact: 273,163 rows, 0 duplicate ids, and per-year near-misses matching Step 1 (541 in 2024, 410 in 2025).

In [ ]:
import pandas as pd, numpy as np, json

df = pd.read_parquet("/content/campaigns_clean.parquet")

print(df.shape, "| duplicate ids:", df["id"].duplicated().sum())
print(df.groupby("year").agg(campaigns=("id", "size"),
                             success_rate=("success", "mean"),
                             near_misses=("near_miss", "sum")).round(2))

(273163, 22) | duplicate ids: 0
      campaigns  success_rate  near_misses
year                                      
2016      36257          0.45          585
2017      34099          0.56          567
2018      29884          0.63          521
2019      28240          0.69          386
2020      25836          0.73          336
2021      23277          0.78          266
2022      20645          0.75          432
2023      21299          0.74          503
2024      26739          0.72          541
2025      26887          0.71          410


### EXPLORATION — Look at the raw JSON columns
Shows the first and last 5 rows. `category` and `location` are long JSON strings, and the old JSON snapshots (2016–18) store them in a different layout from the CSV snapshots — so they must be unpacked before encoding.
Rows are ordered by snapshot date, not launch year, which is why 2023 campaigns appear at the end.

In [ ]:
cols = ["id", "year", "category", "location"]
pd.set_option("display.max_colwidth", 120)   # show more of each JSON string

display(df[cols].head(5))   # first 5 rows
display(df[cols].tail(5))   # last 5 rows

,id,year,category,location
0,1578019152,2016,"{""id"":52,""name"":""Hardware"",""slug"":""technology/hardware"",""urls"":{""web"":{""discover"":""http://www.kickstarter.com/discov...","{""id"":2456572,""name"":""Muscle Shoals"",""slug"":""muscle-shoals-al"",""type"":""Town"",""urls"":{""api"":{""nearby_projects"":""https..."
1,259343731,2016,"{""id"":51,""name"":""Software"",""slug"":""technology/software"",""urls"":{""web"":{""discover"":""http://www.kickstarter.com/discov...","{""id"":2514815,""name"":""Washington"",""slug"":""washington-dc"",""type"":""Town"",""urls"":{""api"":{""nearby_projects"":""https://api..."
2,66578163,2016,"{""id"":362,""name"":""Makerspaces"",""slug"":""technology/makerspaces"",""urls"":{""web"":{""discover"":""http://www.kickstarter.com...","{""id"":2371464,""name"":""Buffalo"",""slug"":""buffalo-ny"",""type"":""Town"",""urls"":{""api"":{""nearby_projects"":""https://api.kicks..."
3,1209091515,2016,"{""id"":28,""name"":""Product Design"",""slug"":""design/product design"",""urls"":{""web"":{""discover"":""http://www.kickstarter.co...","{""id"":2517820,""name"":""West Tisbury"",""slug"":""west-tisbury-ma"",""type"":""Town"",""urls"":{""api"":{""nearby_projects"":""https:/..."
4,1392622950,2016,"{""id"":262,""name"":""Accessories"",""slug"":""fashion/accessories"",""urls"":{""web"":{""discover"":""http://www.kickstarter.com/di...","{""id"":2357024,""name"":""Atlanta"",""slug"":""atlanta-ga"",""type"":""Town"",""urls"":{""api"":{""nearby_projects"":""https://api.kicks..."


,id,year,category,location
273158,15686721,2023,"{""id"":54,""name"":""Mixed Media"",""analytics_name"":""Mixed Media"",""slug"":""art/mixed media"",""position"":6,""parent_id"":1,""pa...","{""id"":2433662,""name"":""Knoxville"",""slug"":""knoxville-tn"",""short_name"":""Knoxville, TN"",""displayable_name"":""Knoxville, T..."
273159,2085816330,2023,"{""id"":54,""name"":""Mixed Media"",""analytics_name"":""Mixed Media"",""slug"":""art/mixed media"",""position"":6,""parent_id"":1,""pa...","{""id"":21125,""name"":""Glasgow"",""slug"":""glasgow-gb"",""short_name"":""Glasgow, UK"",""displayable_name"":""Glasgow, UK"",""locali..."
273160,2087441000,2023,"{""id"":54,""name"":""Mixed Media"",""analytics_name"":""Mixed Media"",""slug"":""art/mixed media"",""position"":6,""parent_id"":1,""pa...","{""id"":2459115,""name"":""New York"",""slug"":""new-york-ny"",""short_name"":""New York, NY"",""displayable_name"":""New York, NY"",""..."
273161,345446786,2023,"{""id"":54,""name"":""Mixed Media"",""analytics_name"":""Mixed Media"",""slug"":""art/mixed media"",""position"":6,""parent_id"":1,""pa...","{""id"":3303,""name"":""North Bay"",""slug"":""north-bay-on-ca"",""short_name"":""North Bay, Canada"",""displayable_name"":""North Ba..."
273162,716012562,2023,"{""id"":54,""name"":""Mixed Media"",""analytics_name"":""Mixed Media"",""slug"":""art/mixed media"",""position"":6,""parent_id"":1,""pa...","{""id"":9807,""name"":""Vancouver"",""slug"":""vancouver-ca"",""short_name"":""Vancouver, Canada"",""displayable_name"":""Vancouver, ..."


### PIPELINE — Unpack category and location into clean labels
Reads each JSON string and adds four columns (originals are kept for checking):
- `subcategory` — e.g. "Tabletop Games" (161 values)
- `parent_category` — part of the slug before "/", e.g. "Games" (15 values)
- `loc_state` — state or region code, e.g. "CA"
- `loc_type` — e.g. "Town"

In [ ]:
# --- Unpack category and location from JSON text into clean labels ---

def parse(s):
    try:
        d = json.loads(s) if isinstance(s, str) else {}
        return d if isinstance(d, dict) else {}
    except ValueError:
        return {}

cat = df["category"].map(parse)
loc = df["location"].map(parse)

df["subcategory"]     = cat.map(lambda d: d.get("name"))
df["parent_category"] = cat.map(lambda d: (d.get("slug") or "").split("/")[0].replace("-", " ").title() or None)
df["loc_state"]       = loc.map(lambda d: d.get("state"))
df["loc_type"]        = loc.map(lambda d: d.get("type"))

# --- Check: distinct values and blanks ---
for col in ["subcategory", "parent_category", "loc_state", "loc_type"]:
    print(f"{col}: {df[col].nunique()} distinct, {df[col].isna().mean():.1%} blank")
print(df["parent_category"].value_counts())

# --- Check: raw JSON next to the clean labels, 5 random rows ---
cols = ["category", "subcategory", "parent_category", "location", "loc_state", "loc_type"]
display(df[cols].sample(5, random_state=1))

subcategory: 161 distinct, 0.0% blank
parent_category: 15 distinct, 0.0% blank
loc_state: 1207 distinct, 0.0% blank
loc_type: 9 distinct, 0.0% blank
parent_category
Games           44011
Technology      29134
Publishing      28212
Film & Video    26984
Art             26503
Design          22915
Comics          21809
Music           20056
Fashion         19132
Food            14398
Crafts           6238
Photography      5223
Theater          4257
Journalism       2848
Dance            1443
Name: count, dtype: int64


,category,subcategory,parent_category,location,loc_state,loc_type
168555,"{""id"":34,""name"":""Tabletop Games"",""analytics_name"":""Tabletop Games"",""slug"":""games/tabletop games"",""position"":7,""paren...",Tabletop Games,Games,"{""id"":2383489,""name"":""Colorado Springs"",""slug"":""colorado-springs-co"",""short_name"":""Colorado Springs, CO"",""displayabl...",CO,Town
196208,"{""id"":46,""name"":""Children's Books"",""analytics_name"":""Children's Books"",""slug"":""publishing/children's books"",""positio...",Children's Books,Publishing,"{""id"":2487796,""name"":""San Antonio"",""slug"":""san-antonio-tx"",""short_name"":""San Antonio, TX"",""displayable_name"":""San An...",TX,Town
36323,"{""id"":28,""name"":""Product Design"",""slug"":""design/product design"",""position"":5,""parent_id"":7,""parent_name"":""Design"",""c...",Product Design,Design,"{""id"":2367105,""name"":""Boston"",""slug"":""boston-ma"",""short_name"":""Boston, MA"",""displayable_name"":""Boston, MA"",""localize...",MA,Town
52102,"{""id"":46,""name"":""Children's Books"",""analytics_name"":""Children's Books"",""slug"":""publishing/children's books"",""positio...",Children's Books,Publishing,"{""id"":2442047,""name"":""Los Angeles"",""slug"":""los-angeles-ca"",""short_name"":""Los Angeles, CA"",""displayable_name"":""Los An...",CA,Town
183540,"{""id"":318,""name"":""Faith"",""analytics_name"":""Faith"",""slug"":""music/faith"",""position"":7,""parent_id"":14,""parent_name"":""Mu...",Faith,Music,"{""id"":2450022,""name"":""Miami"",""slug"":""miami-fl"",""short_name"":""Miami, FL"",""displayable_name"":""Miami, FL"",""localized_na...",FL,Town


### EXPLORATION — How subcategories are spread
Subcategories per parent add up to 171, but there are only 161 distinct names — about 10 names (e.g. Anthologies) appear under more than one parent. This led to the next cell's fix.

In [ ]:
print(df.groupby("parent_category")["subcategory"].nunique())
print(df["subcategory"].value_counts().head(25))

parent_category
Art             14
Comics           6
Crafts          14
Dance            5
Design           8
Fashion          9
Film & Video    20
Food            13
Games            8
Journalism       6
Music           19
Photography      7
Publishing      18
Technology      16
Theater          8
Name: subcategory, dtype: int64
subcategory
Tabletop Games      26628
Product Design      17488
Comic Books         11990
Illustration         8167
Apps                 7725
Fiction              6850
Accessories          6775
Apparel              6706
Playing Cards        6508
Video Games          5879
Art                  5558
Graphic Novels       5463
Children's Books     5156
Gadgets              4646
Web                  3770
Shorts               3568
Nonfiction           3408
Hardware             3283
Drama                3112
Comedy               3093
Documentary          3013
Restaurants          2943
Drinks               2935
Music                2926
Digital Art          2773
Name:

### PIPELINE — Label each subcategory with its parent
Turns "Comedy" into "Film & Video / Comedy" or "Theater / Comedy", so shared names stay separate (171 labels).
Rebuilt from the raw `category` JSON, so it is safe to run more than once.

In [ ]:
df["subcategory"] = df["parent_category"] + " / " + df["category"].map(lambda s: parse(s).get("name"))

print(df["subcategory"].nunique(), "distinct parent / subcategory labels")
print(df["subcategory"].value_counts().head(10))

171 distinct parent / subcategory labels
subcategory
Games / Tabletop Games     26628
Design / Product Design    17488
Comics / Comic Books       11990
Art / Illustration          8167
Technology / Apps           7725
Publishing / Fiction        6850
Fashion / Accessories       6775
Fashion / Apparel           6706
Games / Playing Cards       6508
Games / Video Games         5879
Name: count, dtype: int64


### PIPELINE — Split by launch year
Splits by time, not randomly: the model will be used on future campaigns, so it is tested on the newest ones (L2 slide 12, L6 slide 9; HW4 "Splitting the data").

| Slice | Years | Campaigns | Success | Near-misses |
|---|---|---|---|---|
| Text | 2016–18 | 100,240 | 54% | 1,673 |
| Train | 2019–23 | 119,297 | 73% | 1,923 |
| Validation | 2024 | 26,739 | 72% | 541 |
| Test | 2025 | 26,887 | 71% | 410 |

Slices add up to all 273,163 campaigns. Note: success jumps from 54% to 73% between text and training years, so the blurb model learns from a period with more failures.

In [ ]:
text  = df[df["year"].between(2016, 2018)].copy()   # blurb model only
train = df[df["year"].between(2019, 2023)].copy()   # main models study here
val   = df[df["year"] == 2024].copy()               # make choices here
test  = df[df["year"] == 2025].copy()               # final results, run once

slices = {"text (2016-18)": text, "train (2019-23)": train,
          "val (2024)": val, "test (2025)": test}

for name, s in slices.items():
    print(f"{name:16} {len(s):>7,} campaigns | success {s['success'].mean():.0%} | near-misses {s['near_miss'].sum():>5,}")
print("Total:", sum(len(s) for s in slices.values()), "of", len(df))

text (2016-18)   100,240 campaigns | success 54% | near-misses 1,673
train (2019-23)  119,297 campaigns | success 73% | near-misses 1,923
val (2024)        26,739 campaigns | success 72% | near-misses   541
test (2025)       26,887 campaigns | success 71% | near-misses   410
Total: 273163 of 273163


### PIPELINE — Check no campaign appears in two slices
All six pairs share 0 campaign ids.

In [ ]:
ids = {name: set(s["id"]) for name, s in slices.items()}
names = list(ids)

for i in range(len(names)):
    for j in range(i + 1, len(names)):
        shared = len(ids[names[i]] & ids[names[j]])
        print(f"{names[i]:16} vs {names[j]:16}: {shared} shared")

text (2016-18)   vs train (2019-23) : 0 shared
text (2016-18)   vs val (2024)      : 0 shared
text (2016-18)   vs test (2025)     : 0 shared
train (2019-23)  vs val (2024)      : 0 shared
train (2019-23)  vs test (2025)     : 0 shared
val (2024)       vs test (2025)     : 0 shared


### EXPLORATION — Superseded: top 40 subcategories
Covered 71.4% of training — too much left in "other", so we moved to 60 (next cell).

In [ ]:
top_sub = train["subcategory"].value_counts().head(40).index

covered = train["subcategory"].isin(top_sub).mean()
print(f"Top 40 subcategories cover {covered:.1%} of training campaigns")

for name, s in slices.items():
    print(f"{name:16} {s['subcategory'].isin(top_sub).mean():.1%} covered")

print(list(top_sub))

Top 40 subcategories cover 71.4% of training campaigns
text (2016-18)   62.0% covered
train (2019-23)  71.4% covered
val (2024)       78.3% covered
test (2025)      77.5% covered
['Games / Tabletop Games', 'Design / Product Design', 'Comics / Comic Books', 'Art / Illustration', 'Art / Art', 'Games / Playing Cards', 'Technology / Apps', 'Fashion / Accessories', 'Comics / Graphic Novels', "Publishing / Children's Books", 'Publishing / Fiction', 'Fashion / Apparel', 'Games / Video Games', 'Technology / Gadgets', 'Film & Video / Shorts', 'Art / Digital Art', 'Music / Music', 'Technology / Hardware', 'Film & Video / Drama', 'Food / Drinks', 'Film & Video / Animation', 'Food / Restaurants', 'Publishing / Art Books', 'Publishing / Nonfiction', 'Photography / Photobooks', 'Film & Video / Documentary', 'Art / Mixed Media', 'Technology / Software', 'Technology / Web', 'Music / Country & Folk', 'Design / Toys', 'Film & Video / Horror', 'Film & Video / Comedy', 'Art / Painting', 'Fashion / Jewelry

### PIPELINE — Choose the top 60 subcategories (training only)
Counts subcategories in 2019–23 only, so later years don't shape the inputs (HW4 `common_models`).
Covers 81.1% of training (validation/test about 85%); the 20 labels added after 40 average about 580 campaigns each.

In [ ]:
top_sub = train["subcategory"].value_counts().head(60).index

covered = train["subcategory"].isin(top_sub).mean()
print(f"Top 60 subcategories cover {covered:.1%} of training campaigns")

for name, s in slices.items():
    print(f"{name:16} {s['subcategory'].isin(top_sub).mean():.1%} covered")

print(list(top_sub))

Top 60 subcategories cover 81.1% of training campaigns
text (2016-18)   73.9% covered
train (2019-23)  81.1% covered
val (2024)       85.3% covered
test (2025)      85.1% covered
['Games / Tabletop Games', 'Design / Product Design', 'Comics / Comic Books', 'Art / Illustration', 'Art / Art', 'Games / Playing Cards', 'Technology / Apps', 'Fashion / Accessories', 'Comics / Graphic Novels', "Publishing / Children's Books", 'Publishing / Fiction', 'Fashion / Apparel', 'Games / Video Games', 'Technology / Gadgets', 'Film & Video / Shorts', 'Art / Digital Art', 'Music / Music', 'Technology / Hardware', 'Film & Video / Drama', 'Food / Drinks', 'Film & Video / Animation', 'Food / Restaurants', 'Publishing / Art Books', 'Publishing / Nonfiction', 'Photography / Photobooks', 'Film & Video / Documentary', 'Art / Mixed Media', 'Technology / Software', 'Technology / Web', 'Music / Country & Folk', 'Design / Toys', 'Film & Video / Horror', 'Film & Video / Comedy', 'Art / Painting', 'Fashion / Jewelry

### PIPELINE — Group the remaining subcategories as "other"
Every slice keeps only the training top 60; the rest become "other" (18.9% of training). Those campaigns still carry their parent category.

In [ ]:
for s in slices.values():
    s["subcategory"] = s["subcategory"].where(s["subcategory"].isin(top_sub), "other")

for name, s in slices.items():
    print(f"{name:16} {s['subcategory'].nunique()} labels | {(s['subcategory'] == 'other').mean():.1%} other")

text (2016-18)   60 labels | 26.1% other
train (2019-23)  61 labels | 18.9% other
val (2024)       61 labels | 14.7% other
test (2025)      61 labels | 14.9% other


### EXPLORATION — Superseded: top 20 states, no country prefix
Covered only 58%, and codes clash across countries ("WA" = Washington or Western Australia; "CA" = California or Canada). Led to the country + state labels below.

In [ ]:
top_state = train["loc_state"].value_counts().head(20).index

for name, s in slices.items():
    print(f"{name:16} {s['loc_state'].isin(top_state).mean():.1%} covered")

print(list(top_state))

text (2016-18)   59.5% covered
train (2019-23)  58.0% covered
val (2024)       55.0% covered
test (2025)      54.8% covered
['England', 'CA', 'NY', 'TX', 'FL', 'ON', 'WA', 'IL', 'Hong Kong Island', 'PA', 'GA', 'OH', 'OR', 'CO', 'MA', 'NC', 'MI', 'VA', 'BC', 'TN']


### EXPLORATION — Superseded: country + state, top 40
Covered 73.2% of training; we moved to 60 (next cell).

In [ ]:
for s in [df] + list(slices.values()):
    s["loc_state"] = s["country"] + " / " + s["location"].map(lambda x: parse(x).get("state"))

top_state = train["loc_state"].value_counts().head(40).index

for name, s in slices.items():
    print(f"{name:16} {s['loc_state'].isin(top_state).mean():.1%} covered")
print(list(top_state))

text (2016-18)   74.6% covered
train (2019-23)  73.2% covered
val (2024)       68.9% covered
test (2025)      69.0% covered
['GB / England', 'US / CA', 'US / NY', 'US / TX', 'US / FL', 'CA / ON', 'US / WA', 'US / IL', 'HK / Hong Kong Island', 'US / PA', 'US / GA', 'US / OH', 'US / OR', 'US / CO', 'US / MA', 'US / NC', 'US / MI', 'US / VA', 'US / TN', 'CA / BC', 'US / UT', 'US / AZ', 'US / NJ', 'GB / Scotland', 'US / MN', 'FR / Ile-de-France', 'AU / NSW', 'SG / Central Singapore', 'US / MO', 'US / NV', 'CA / QC', 'AU / VIC', 'US / MD', 'US / IN', 'ES / Madrid', 'US / WI', 'CA / AB', 'JP / Tokyo Prefecture', 'US / CT', 'IT / Lombardy']


### PIPELINE — Country + state labels, top 60 (training only)
Adds the country so codes don't clash — "US / WA" vs "AU / WA", "US / CA" vs Canada.
Covers 81.2% of training (validation/test about 77%); the 20 places added after 40 average about 480 campaigns each.
Rebuilt from the raw columns, so it is safe to run more than once.

In [ ]:
for s in [df] + list(slices.values()):
    s["loc_state"] = s["country"] + " / " + s["location"].map(lambda x: parse(x).get("state"))

top_state = train["loc_state"].value_counts().head(60).index

for name, s in slices.items():
    print(f"{name:16} {s['loc_state'].isin(top_state).mean():.1%} covered")
print(list(top_state))

text (2016-18)   83.1% covered
train (2019-23)  81.2% covered
val (2024)       77.3% covered
test (2025)      77.7% covered
['GB / England', 'US / CA', 'US / NY', 'US / TX', 'US / FL', 'CA / ON', 'US / WA', 'US / IL', 'HK / Hong Kong Island', 'US / PA', 'US / GA', 'US / OH', 'US / OR', 'US / CO', 'US / MA', 'US / NC', 'US / MI', 'US / VA', 'US / TN', 'CA / BC', 'US / UT', 'US / AZ', 'US / NJ', 'GB / Scotland', 'US / MN', 'FR / Ile-de-France', 'AU / NSW', 'SG / Central Singapore', 'US / MO', 'US / NV', 'CA / QC', 'AU / VIC', 'US / MD', 'US / IN', 'ES / Madrid', 'US / WI', 'CA / AB', 'JP / Tokyo Prefecture', 'US / CT', 'IT / Lombardy', 'DE / Berlin', 'US / SC', 'US / KY', 'ES / Catalonia', 'US / DC', 'MX / Distrito Federal', 'AU / QLD', 'DE / North Rhine-Westphalia', 'IT / Piedmont', 'DE / Bavaria', 'MX / Baja California', 'US / AL', 'SE / Stockholm', 'US / DE', 'FR / Auvergne', 'US / ID', 'US / LA', 'US / OK', 'NL / North Holland', 'US / NM']


### PIPELINE — Group the remaining locations as "other"
Keeps the training top 60; the rest become "other" (18.8% of training, about 22.5% of validation/test). Those campaigns still carry `country`.

In [ ]:
for s in slices.values():
    s["loc_state"] = s["loc_state"].where(s["loc_state"].isin(top_state), "other")

for name, s in slices.items():
    print(f"{name:16} {s['loc_state'].nunique()} labels | {(s['loc_state'] == 'other').mean():.1%} other")

text (2016-18)   61 labels | 16.9% other
train (2019-23)  61 labels | 18.8% other
val (2024)       61 labels | 22.7% other
test (2025)      61 labels | 22.3% other


### EXPLORATION — Check country, currency and location type
- `country`: 25 values, none unseen in validation/test → keep all.
- `currency`: repeats country (e.g. US and USD both 68,284) → dropped.
- `loc_type`: Town is 94% of training → group the rare types (next cell).

In [ ]:
for col in ["country", "currency", "loc_type"]:
    counts = train[col].value_counts()
    print(f"\n{col}: {len(counts)} distinct in training")
    print(counts.head(25))
    for name, s in slices.items():
        unseen = (~s[col].isin(counts.index)).sum()
        print(f"  {name:16} {unseen} campaigns with a value not seen in training")


country: 25 distinct in training
country
US    68284
GB    15498
CA     6764
DE     3399
AU     3243
FR     2816
HK     2809
IT     2672
ES     2601
MX     2599
JP     1312
NL     1113
SG     1075
SE     1034
CH      618
DK      595
PL      490
NZ      486
BE      480
IE      423
AT      397
NO      241
GR      229
SI       69
LU       50
Name: count, dtype: Int64
  text (2016-18)   0 campaigns with a value not seen in training
  train (2019-23)  0 campaigns with a value not seen in training
  val (2024)       0 campaigns with a value not seen in training
  test (2025)      0 campaigns with a value not seen in training

currency: 15 distinct in training
currency
USD    68284
GBP    15498
EUR    15045
CAD     6764
AUD     3243
HKD     2809
MXN     2599
JPY     1312
SGD     1075
SEK      849
CHF      533
DKK      486
NZD      486
NOK      178
PLN      136
Name: count, dtype: Int64
  text (2016-18)   0 campaigns with a value not seen in training
  train (2019-23)  0 campaigns with a valu

### PIPELINE — Group rare location types as "other"
Keeps Town, County, Suburb and LocalAdmin (each 1,199+ campaigns); Zip, Island, Country and Miscellaneous become "other" (315 campaigns).

In [ ]:
top_type = ["Town", "County", "Suburb", "LocalAdmin"]
for s in slices.values():
    s["loc_type"] = s["loc_type"].where(s["loc_type"].isin(top_type), "other")
print(train["loc_type"].value_counts())

loc_type
Town          112125
County          3001
Suburb          2657
LocalAdmin      1199
other            315
Name: count, dtype: int64


### PIPELINE — Campaign length in days
(deadline − launch) ÷ 86,400 seconds. Dates are stored as Unix time (seconds since Jan 1, 1970).
Training pattern: shorter campaigns succeed more — 88% at ≤20 days vs 41% at 50–60 days. A link, not proven cause.
Note: 2,083 training campaigns run over 60 days (max 120.6); kept, flagged for the README.

In [ ]:
for s in slices.values():
    launched = pd.to_numeric(s["launched_at"])
    deadline = pd.to_numeric(s["deadline"])
    s["duration_days"] = (deadline - launched) / 86400   # seconds → days

print(train["duration_days"].describe().round(1))
print(train.groupby(pd.cut(train["duration_days"], [0, 20, 30, 40, 50, 60, 100]),
                    observed=True)["success"].agg(["size", "mean"]).round(2))

count    119297.0
mean         32.6
std          12.9
min           1.0
25%          28.0
50%          30.0
75%          35.0
max         120.6
Name: duration_days, dtype: float64
                size  mean
duration_days             
(0, 20]        15875  0.88
(20, 30]       57053  0.77
(30, 40]       22393  0.79
(40, 50]        8104  0.66
(50, 60]       13789  0.41
(60, 100]       2082  0.39


### PIPELINE — Prep time (page creation → launch)
Training pattern: campaigns launched within a week of creating the page succeed 59% of the time vs 77–81% with more prep; little gain after about a month.
Values run from 0 days to nearly 11 years, so the next cell puts them on a log scale.

In [ ]:
for s in slices.values():
    created  = pd.to_numeric(s["created_at"])
    launched = pd.to_numeric(s["launched_at"])
    s["prep_days"] = (launched - created) / 86400

print(train["prep_days"].describe().round(1))
print(train.groupby(pd.cut(train["prep_days"], [-1, 7, 30, 90, 365, 10000]),
                    observed=True)["success"].agg(["size", "mean"]).round(2))

count    119297.0
mean         63.6
std         174.2
min           0.0
25%           6.6
50%          19.2
75%          53.0
max        3959.1
Name: prep_days, dtype: float64
               size  mean
prep_days                
(-1, 7]       31437  0.59
(7, 30]       42268  0.77
(30, 90]      27300  0.81
(90, 365]     14763  0.80
(365, 10000]   3529  0.75


### PIPELINE — Log scale for prep time
A log measures "how many times longer," not "how many days more" — 1 → 10 days is a big step, 1 year → 1 year 9 days is not. Logistic regression needs this; boosting is unaffected.
`log1p` = log(1 + x), so 0 days doesn't break. The original `prep_days` stays in the files for reading but is not a model input.

In [ ]:
for s in slices.values():
    s["log_prep_days"] = np.log1p(s["prep_days"])

print(train["log_prep_days"].describe().round(2))

count    119297.00
mean          3.01
std           1.49
min           0.00
25%           2.03
50%           3.01
75%           3.99
max           8.28
Name: log_prep_days, dtype: float64


### PIPELINE — Launch month and weekday
Training pattern: December 65% (other months 72–76%); Tuesday 80%, weekends 66–68%.
Kept as labels, not numbers — December (12) isn't "more" than January (1).
Note: dates are UTC, so some US evening launches count as the next day.

In [ ]:
for s in slices.values():
    t = pd.to_datetime(pd.to_numeric(s["launched_at"]), unit="s")
    s["launch_month"]   = t.dt.month          # 1 = Jan … 12 = Dec
    s["launch_weekday"] = t.dt.day_name()     # Monday … Sunday

for col in ["launch_month", "launch_weekday"]:
    print(train.groupby(col)["success"].agg(["size", "mean"]).round(2), "\n")

               size  mean
launch_month             
1             10102  0.73
2             10733  0.76
3             11141  0.75
4              9675  0.74
5             10417  0.72
6              9755  0.74
7              9745  0.72
8             10019  0.73
9             10350  0.75
10            11692  0.75
11             9480  0.74
12             6188  0.65 

                 size  mean
launch_weekday             
Friday          17418  0.73
Monday          18222  0.72
Saturday         9773  0.68
Sunday           7667  0.66
Thursday        17196  0.71
Tuesday         30311  0.80
Wednesday       18710  0.72 



### PIPELINE — Name and blurb length (characters)
Name length: success rises from 64% (≤20 characters) to 79% (52–61).
Blurb length: flat (71–76%); kept because it costs one column. What the blurb says is Model 4's job.

In [ ]:
for s in slices.values():
    s["name_len"]  = s["name"].fillna("").str.len()
    s["blurb_len"] = s["blurb"].fillna("").str.len()

for col in ["name_len", "blurb_len"]:
    print(train[col].describe().round(1))
    print(train.groupby(pd.qcut(train[col], 5, duplicates="drop"), observed=True)["success"]
               .agg(["size", "mean"]).round(2), "\n")

count    119297.0
mean         35.0
std          15.5
min           1.0
25%          22.0
50%          35.0
75%          48.0
max          61.0
Name: name_len, dtype: Float64
                size  mean
name_len                  
(0.999, 20.0]  26025  0.64
(20.0, 30.0]   23943  0.73
(30.0, 40.0]   23240  0.75
(40.0, 51.0]   22402  0.77
(51.0, 61.0]   23687  0.79 

count    119297.0
mean         92.9
std          34.5
min           0.0
25%          65.0
50%          99.0
75%         125.0
max         370.0
Name: blurb_len, dtype: Float64
                 size  mean
blurb_len                  
(-0.001, 58.0]  24319  0.72
(58.0, 86.0]    23931  0.76
(86.0, 111.0]   23685  0.75
(111.0, 129.0]  25238  0.72
(129.0, 370.0]  22124  0.71 



### EXPLORATION — Before vs after
Five random training campaigns with original columns beside the prepared ones (JSON → labels, timestamps → days, goal → log goal).

In [ ]:
show = ["id", "year",
        # labels: original → prepared
        "category", "parent_category", "subcategory",
        "location", "country", "loc_state", "loc_type",
        # numbers: original → prepared
        "goal_usd", "log_goal_usd",
        "launched_at", "deadline", "duration_days",
        "created_at", "prep_days", "log_prep_days",
        "launch_month", "launch_weekday",
        "name", "name_len", "blurb_len",
        # answers
        "success", "near_miss"]

pd.set_option("display.max_columns", None)    # show every column
pd.set_option("display.max_colwidth", 40)     # shorten the long JSON text
display(train[show].sample(5, random_state=7))

,id,year,category,parent_category,subcategory,location,country,loc_state,loc_type,goal_usd,log_goal_usd,launched_at,deadline,duration_days,created_at,prep_days,log_prep_days,launch_month,launch_weekday,name,name_len,blurb_len,success,near_miss
249650,1233482554,2022,"{""id"":53,""name"":""Public Art"",""analyt...",Art,other,"{""id"":2364363,""name"":""Binghamton"",""s...",US,US / NY,Town,35000.00000,10.463132,1659982706,1662436740,28.403171,1659313373,7.746910,2.168700,8,Monday,LUMA Projection Arts Festival: Year 8,37,98,1,0
127803,591874456,2023,"{""id"":14,""name"":""Music"",""analytics_n...",Music,Music / Music,"{""id"":554890,""name"":""Copenhagen"",""sl...",DK,other,Town,3324.72268,8.109442,1679995997,1682595360,30.085220,1674563551,62.875532,4.156936,3,Tuesday,"Support my new album ""Occasionally, ...",58,91,1,0
61348,481668155,2019,"{""id"":45,""name"":""Art Books"",""analyti...",Publishing,Publishing / Art Books,"{""id"":2437673,""name"":""Lebanon"",""slug...",US,other,Town,3000.00000,8.006701,1556136012,1561320012,60.000000,1543518021,146.041562,4.990715,4,Wednesday,Raq-Box: Help us create a box of goo...,56,53,0,0
116985,1229760678,2023,"{""id"":54,""name"":""Mixed Media"",""analy...",Art,Art / Mixed Media,"{""id"":4118,""name"":""Toronto"",""slug"":""...",CA,CA / ON,Town,2192.79813,7.693390,1679495374,1682892000,39.312801,1679335955,1.845127,1.045608,3,Wednesday,Evolving Connection Group Show,30,64,0,0
220280,1797438762,2022,"{""id"":299,""name"":""Music Videos"",""ana...",Film & Video,other,"{""id"":44418,""name"":""London"",""slug"":""...",GB,GB / England,Town,3935.81991,8.278129,1646908766,1650927600,46.514282,1642601744,49.849792,3.928876,3,Thursday,Burr Island - New Music Vids,28,90,1,0


### PIPELINE — One-hot encode, fill blanks, line up columns (HW4 `build_features()`)
- Numbers (5): `log_goal_usd`, `duration_days`, `log_prep_days`, `name_len`, `blurb_len`. Blanks get the **training** median (there were 0).
- Labels → yes/no columns (186): `parent_category` 15, `subcategory` 61, `loc_state` 61, `loc_type` 5, `country` 25, `launch_month` 12, `launch_weekday` 7.
- `.reindex(columns=X_train.columns, fill_value=0)` gives validation and test exactly the training columns.

Result: 191 columns and 0 blanks in all three slices; columns identical. Left out on purpose: `currency`, `goal_usd`, `prep_days`, raw JSON/text/dates, every `post_` column. The text slice isn't encoded.

In [ ]:
NUMERIC = ["log_goal_usd", "duration_days", "log_prep_days", "name_len", "blurb_len"]
CATEGORICAL = ["parent_category", "subcategory", "loc_state", "loc_type",
               "country", "launch_month", "launch_weekday"]

print("Blanks in training numbers:\n", train[NUMERIC].isna().sum(), "\n")

def build_features(df, medians=None):
    X = df[NUMERIC].astype(float)
    if medians is None:                 # training: learn the medians
        medians = X.median()
    X = X.fillna(medians)               # all slices: fill with TRAINING medians
    dummies = pd.get_dummies(df[CATEGORICAL].astype(str)).astype(int)
    return pd.concat([X, dummies], axis=1), medians

X_train, medians = build_features(train)
X_val,  _ = build_features(val,  medians)
X_test, _ = build_features(test, medians)
X_val  = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train, y_val, y_test = train["success"].values, val["success"].values, test["success"].values

for name, X in [("train", X_train), ("val", X_val), ("test", X_test)]:
    print(f"{name:5} {X.shape} | blanks left: {X.isna().sum().sum()}")
print("Same columns in all three:", list(X_train.columns) == list(X_val.columns) == list(X_test.columns))

Blanks in training numbers:
 log_goal_usd     0
duration_days    0
log_prep_days    0
name_len         0
blurb_len        0
dtype: int64 

train (119297, 191) | blanks left: 0
val   (26739, 191) | blanks left: 0
test  (26887, 191) | blanks left: 0
Same columns in all three: True


### PIPELINE — Save and download `split.zip`
Seven files every later step loads:
- `X_train` / `X_val` / `X_test` — 191 encoded inputs, for the models
- `slice_text` / `slice_train` / `slice_val` / `slice_test` — full rows (`id`, `success`, `near_miss`, blurbs), for the blurb model and grading

Row *n* of an `X_` file is the same campaign as row *n* of the matching `slice_` file.
Open decision: near-misses (541 / 410) are below the ~1,000 target. The split is locked; if hit rates look noisy, widen the near-miss range later — that changes one line and rebuilds no model.

In [ ]:
import os
os.makedirs("/content/split", exist_ok=True)

def save(df, name):
    out = df.copy()
    for c in out.select_dtypes("object").columns:
        out[c] = out[c].astype("string")      # avoids mixed-type save errors
    out.to_parquet(f"/content/split/{name}.parquet", index=False)

# model inputs (191 columns each)
for name, X in [("X_train", X_train), ("X_val", X_val), ("X_test", X_test)]:
    save(X, name)

# full slices: id, answers (success, near_miss), blurbs and readable columns
for name, s in [("text", text), ("train", train), ("val", val), ("test", test)]:
    save(s, f"slice_{name}")

!cd /content && zip -rq split.zip split && ls -lh split split.zip
from google.colab import files
files.download("/content/split.zip")

-rw-r--r-- 1 root root  70M Nov  7 20:00 split.zip

split:
total 87M
-rw-r--r-- 1 root root 8.1M Nov  7 20:00 slice_test.parquet
-rw-r--r-- 1 root root  31M Nov  7 20:00 slice_text.parquet
-rw-r--r-- 1 root root  36M Nov  7 20:00 slice_train.parquet
-rw-r--r-- 1 root root 8.0M Nov  7 20:00 slice_val.parquet
-rw-r--r-- 1 root root 872K Nov  7 20:00 X_test.parquet
-rw-r--r-- 1 root root 3.3M Nov  7 20:00 X_train.parquet
-rw-r--r-- 1 root root 873K Nov  7 20:00 X_val.parquet


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>